# 环节 04 · 网络隔离与出口白名单演示

纯 Python 标准库，零依赖。手搓四件事：

1. 域名白名单匹配（精确 / 前导 `*.` / 裸 `*`）；
2. IPv6 括号写法；
3. 出站决策（netns 状态 + 白名单 + 私网拒绝）；
4. 绕过检测（IP 直连 / domain fronting / 通配过宽）。

In [ ]:
# §1 白名单匹配：只认两种通配符（前导 *. 与裸 *）
def match(rule, host):
    if rule == "*":
        return True
    if rule.startswith("*."):
        return host.endswith(rule[1:]) and host != rule[2:]
    return host == rule


RULES = ["github.com", "*.npmjs.org", "*"]
TESTS = ["github.com", "api.github.com", "registry.npmjs.org", "evil.com"]

print(f"{'host':<22}{'匹配规则'}")
print("-" * 40)
for h in TESTS:
    hit = next((r for r in RULES if match(r, h)), None)
    print(f"{h:<22}{hit}")
print()
print("危险示例：如果是 'example.*'（TLD 级通配）—— 语义上就不安全，多数实现不支持")
print("最危险的是裸 '*'：等于没有出口管控")

In [ ]:
# §2 IPv6：需要括号写法，且不同配置项规范形式可能不同
def norm_host(entry):
    return entry.strip()


ENTRIES = ["[::1]", "[::1]:443", "example.com", "::1"]
for e in ENTRIES:
    kind = "IPv6(括号)" if e.startswith("[") else ("IPv6(裸)" if ":" in e else "IPv4/域名")
    print(f"{e:<14} → {kind}")
print()
print("坑：允许列表用 '[::1]'、注入列表用 '::1'（两种匹配器不同）")
print("    拼写不一致 → 规则看似配了却不生效")

In [ ]:
# §3 出站决策器
PRIVATE_CIDRS = ["127.", "10.", "192.168.", "169.254.", "172.16."]


def is_private(host):
    return any(host.startswith(p) for p in PRIVATE_CIDRS)


def decide(host, netns_isolated, allowlist, direct_ip=False):
    if netns_isolated:
        return "BLOCK: 无网（netns 只有 lo）"
    if direct_ip:
        return "BLOCK: 禁止 IP 直连（出口强制经代理）"
    if is_private(host):
        return "BLOCK: 私网/链路本地地址"
    if not any(match(r, host) for r in allowlist):
        return "BLOCK: 不在白名单"
    return "ALLOW"


ALLOW = ["github.com", "*.npmjs.org"]
CASES = [
    ("registry.npmjs.org", False, False),
    ("evil.com",           False, False),
    ("169.254.169.254",    False, False),
    ("10.0.0.5",           False, False),
    ("github.com",         True,  False),
    ("github.com",         False, True),
]
for host, iso, direct in CASES:
    print(f"{host:<20} {decide(host, iso, ALLOW, direct)}")

In [ ]:
# §4 白名单拦得住什么、拦不住什么
print("白名单能拦：")
print("  - 不在白名单的目的地（evil.com / 陌生域名）")
print("  - IP 直连（若出口强制走代理）")
print("  - 私网与云元数据地址（若显式拒绝）")
print()
print("白名单拦不住（需要 TLS 终止 + 内容检查才可能拦）：")
print("  - 在合法主机上上传数据（合法主机的滥用）")
print("  - domain fronting（SNI 与 Host 不一致，借白名单域名连接别的后端）")
print("  - DNS 隧道（若允许自由解析）")
print()
print("结论：白名单拦的是『目的地』，拦不住『意图』")

## §5 自测表

| # | 问题 | 答案要点 |
|---|---|---|
| 1 | 新建 net namespace 后能上网吗？ | 不能；只有一张 down 的 lo |
| 2 | 改 `/etc/hosts` 算网络隔离吗？ | 不算；IP 直连可绕过 |
| 3 | 白名单最大盲区？ | 不检查内容，白名单主机可当外传通道 |
| 4 | 代理为什么必须放沙箱外？ | 否则可被绕过；也不能让沙箱拿到代理实现 |
| 5 | 为什么必须拒绝 `169.254.169.254`？ | 云实例元数据能拿临时凭证（SSRF 经典一步） |
| 6 | Unix socket 为什么是"第二张网"？ | 能调高权限本地服务，`docker.sock` 一步逃逸 |

**相关长文**：[环节04-网络隔离与出口管控详解.md](./环节04-网络隔离与出口管控详解.md)